# ==========================================
# ТЕСТОВОЕ ЗАДАНИЕ DATA ANALYST
# Расчёт коэффициентов пролонгации договоров за 2023 год
# ==========================================
# Автор: Еремина Анна Владимировна
# Дата: 03.05.2026г.
# ==========================================

=======================================
##  Шаг 1. Импорт библиотек и настройка окружения, загрузка данных
=======================================

In [10]:
import pandas as pd
import numpy as np
import re
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import LineChart, Reference, BarChart
from openpyxl.drawing.image import Image as XLImage
from datetime import datetime
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("🔄 ЗАГРУЗКА И ОБРАБОТКА ДАННЫХ...")
print("=" * 60)

# Загружаем данные
prolong_id = '1lUfXK-WML9LlNjzmI9IncQWLsM_v21wx'
fin_id = '1kSg3Ln2L7KyWk2ccniphlsqDuBZD4ytQ'

prolong_url = f'https://drive.google.com/uc?id={prolong_id}'
fin_url = f'https://drive.google.com/uc?id={fin_id}'

prolong = pd.read_csv(prolong_url)
fin = pd.read_csv(fin_url)

🔄 ЗАГРУЗКА И ОБРАБОТКА ДАННЫХ...


In [11]:
# Словарь месяцев
month_mapping = {
    'Ноябрь 2022': '2022-11', 'Декабрь 2022': '2022-12',
    'Январь 2023': '2023-01', 'Февраль 2023': '2023-02',
    'Март 2023': '2023-03', 'Апрель 2023': '2023-04',
    'Май 2023': '2023-05', 'Июнь 2023': '2023-06',
    'Июль 2023': '2023-07', 'Август 2023': '2023-08',
    'Сентябрь 2023': '2023-09', 'Октябрь 2023': '2023-10',
    'Ноябрь 2023': '2023-11', 'Декабрь 2023': '2023-12',
    'Январь 2024': '2024-01', 'Февраль 2024': '2024-02'
}

In [12]:
def clean_amount(value):
    if pd.isna(value):
        return np.nan
    value_str = str(value).strip()
    if value_str in ['в ноль', 'стоп', 'end', '']:
        return np.nan
    value_str = value_str.replace(' ', '')
    if ',' in value_str:
        parts = value_str.split(',')
        if len(parts) == 2 and len(parts[1]) == 2 and parts[1].isdigit():
            value_str = value_str.replace(',', '.')
        else:
            value_str = value_str.replace(',', '')
    value_str = re.sub(r'[^\d.-]', '', value_str)
    try:
        return float(value_str)
    except:
        return np.nan

# =================================================================
# ШАГ 2: ПРЕДОБРАБОТКА ДАННЫХ
# ==========================================

In [13]:
fin_copy = fin.copy()
month_cols = [c for c in fin_copy.columns if any(m in c for m in month_mapping.keys())]
for col in month_cols:
    fin_copy[col] = fin_copy[col].apply(clean_amount)
fin_copy.rename(columns=month_mapping, inplace=True)
prolong['month_std'] = prolong['month'].apply(lambda x: month_mapping.get(x.strip().title(), None))

all_months = sorted([c for c in fin_copy.columns if re.match(r'\d{4}-\d{2}', c)])

# ============================================================================
# ШАГ 3: СОЗДАНИЕ СПРАВОЧНИКА ПРОЕКТОВ
# ==========================================

In [14]:
project_last_month = {}
project_manager = {}

for _, row in prolong.dropna(subset=['month_std']).iterrows():
    pid, month, am = row['id'], row['month_std'], row['AM']
    if pid not in project_last_month or month > project_last_month[pid]:
        project_last_month[pid] = month
        project_manager[pid] = am

for pid in fin_copy['id'].unique():
    if pid not in project_manager:
        mask = fin_copy['id'] == pid
        if mask.any():
            am = fin_copy.loc[mask, 'Account'].dropna()
            if not am.empty:
                project_manager[pid] = am.iloc[0]
            else:
                project_manager[pid] = 'без А/М'

def get_shipment(pid, month):
    proj_data = fin_copy[fin_copy['id'] == pid]
    if month in proj_data.columns:
        val = proj_data[month].iloc[0]
        if isinstance(val, (int, float)) and not pd.isna(val):
            return val
    return 0

print(f"✅ Загружено проектов: {len(project_last_month)}")
print(f"✅ Менеджеров: {len(set(project_manager.values()))}")

✅ Загружено проектов: 313
✅ Менеджеров: 10


# ============================================================================
# ШАГ 4: РАСЧЁТ КОЭФФИЦИЕНТОВ
# ==========================================


In [15]:
months_2023 = ['Январь 2023', 'Февраль 2023', 'Март 2023', 'Апрель 2023',
               'Май 2023', 'Июнь 2023', 'Июль 2023', 'Август 2023',
               'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023', 'Декабрь 2023']

month_std_map = {
    'Январь 2023': '2023-01', 'Февраль 2023': '2023-02', 'Март 2023': '2023-03',
    'Апрель 2023': '2023-04', 'Май 2023': '2023-05', 'Июнь 2023': '2023-06',
    'Июль 2023': '2023-07', 'Август 2023': '2023-08', 'Сентябрь 2023': '2023-09',
    'Октябрь 2023': '2023-10', 'Ноябрь 2023': '2023-11', 'Декабрь 2023': '2023-12'
}

monthly_data = []
manager_monthly_k1 = {m: {} for m in months_2023}
manager_monthly_k2 = {m: {} for m in months_2023}
manager_yearly = {}

for month_name in months_2023:
    month_std = month_std_map[month_name]
    year, month_num = map(int, month_std.split('-'))

    if month_num == 1:
        prev_month_std = f"{year-1}-12"
        prev_2_month_std = f"{year-1}-11"
    elif month_num == 2:
        prev_month_std = f"{year}-01"
        prev_2_month_std = f"{year-1}-12"
    else:
        prev_month_std = f"{year}-{month_num-1:02d}"
        prev_2_month_std = f"{year}-{month_num-2:02d}"

    k1_base_sum, k1_renewed_sum = 0, 0
    k2_base_sum, k2_renewed_sum = 0, 0
    manager_k1_base, manager_k1_renewed = {}, {}
    manager_k2_base, manager_k2_renewed = {}, {}

    for pid, last_month in project_last_month.items():
        am = project_manager.get(pid, 'без А/М')

        # Проверка на стоп
        proj_data = fin_copy[fin_copy['id'] == pid]
        is_stopped = False
        for _, row in proj_data.iterrows():
            for col in all_months:
                val = row[col]
                if isinstance(val, str) and val.lower() in ['стоп', 'end']:
                    if col <= last_month:
                        is_stopped = True
                        break
            if is_stopped:
                break
        if is_stopped:
            continue

        base_amount = get_shipment(pid, last_month)
        if base_amount == 0:
            last_idx = all_months.index(last_month)
            if last_idx > 0:
                base_amount = get_shipment(pid, all_months[last_idx - 1])
        if base_amount <= 0:
            continue

        # K1
        if last_month == prev_month_std:
            renewal = get_shipment(pid, month_std)
            k1_base_sum += base_amount
            if renewal > 0:
                k1_renewed_sum += renewal

            if am not in manager_k1_base:
                manager_k1_base[am] = 0
                manager_k1_renewed[am] = 0
            manager_k1_base[am] += base_amount
            if renewal > 0:
                manager_k1_renewed[am] += renewal

        # K2
        if last_month == prev_2_month_std:
            has_prev = get_shipment(pid, prev_month_std) > 0
            if not has_prev:
                renewal = get_shipment(pid, month_std)
                k2_base_sum += base_amount
                if renewal > 0:
                    k2_renewed_sum += renewal

                if am not in manager_k2_base:
                    manager_k2_base[am] = 0
                    manager_k2_renewed[am] = 0
                manager_k2_base[am] += base_amount
                if renewal > 0:
                    manager_k2_renewed[am] += renewal

    k1 = k1_renewed_sum / k1_base_sum if k1_base_sum > 0 else 0
    k2 = k2_renewed_sum / k2_base_sum if k2_base_sum > 0 else 0

    monthly_data.append({
        'Месяц': month_name,
        'K1_база': k1_base_sum,
        'K1_пролонгировано': k1_renewed_sum,
        'K1': k1,
        'K2_база': k2_base_sum,
        'K2_пролонгировано': k2_renewed_sum,
        'K2': k2
    })

    for am in manager_k1_base:
        k1_val = manager_k1_renewed.get(am, 0) / manager_k1_base[am] if manager_k1_base[am] > 0 else 0
        manager_monthly_k1[month_name][am] = k1_val

        if am not in manager_yearly:
            manager_yearly[am] = {'k1_base': 0, 'k1_renewed': 0, 'k2_base': 0, 'k2_renewed': 0}
        manager_yearly[am]['k1_base'] += manager_k1_base[am]
        manager_yearly[am]['k1_renewed'] += manager_k1_renewed.get(am, 0)

    for am in manager_k2_base:
        k2_val = manager_k2_renewed.get(am, 0) / manager_k2_base[am] if manager_k2_base[am] > 0 else 0
        manager_monthly_k2[month_name][am] = k2_val

        if am not in manager_yearly:
            manager_yearly[am] = {'k1_base': 0, 'k1_renewed': 0, 'k2_base': 0, 'k2_renewed': 0}
        manager_yearly[am]['k2_base'] += manager_k2_base[am]
        manager_yearly[am]['k2_renewed'] += manager_k2_renewed.get(am, 0)

df_monthly = pd.DataFrame([{
    'Месяц': d['Месяц'],
    'coef1_den': d['K1_база'],
    'coef1_num': d['K1_пролонгировано'],
    'coef1': d['K1'],
    'coef2_den': d['K2_база'],
    'coef2_num': d['K2_пролонгировано'],
    'coef2': d['K2']
} for d in monthly_data])

# ============================================================================
# ШАГ 5: РАСЧЁТ ПО МЕНЕДЖЕРАМ
# ==========================================

In [16]:
manager_list = []
for am, stats in manager_yearly.items():
    k1 = stats['k1_renewed'] / stats['k1_base'] if stats['k1_base'] > 0 else 0
    k2 = stats['k2_renewed'] / stats['k2_base'] if stats['k2_base'] > 0 else 0
    manager_list.append({
        'Менеджер': am,
        'coef1_for': stats['k1_base'],
        'coef1_ren': stats['k1_renewed'],
        'coef1': k1,
        'coef2_for': stats['k2_base'],
        'coef2_ren': stats['k2_renewed'],
        'coef2': k2
    })

df_mgr_yearly = pd.DataFrame(manager_list).sort_values('coef1', ascending=False)

print(f"✅ Менеджеров в отчете: {len(df_mgr_yearly)}")
print(df_mgr_yearly[['Менеджер', 'coef1', 'coef2']].head(10))


✅ Менеджеров в отчете: 8
                        Менеджер     coef1     coef2
7        Петрова Анна Дмитриевна  1.111182  0.000000
5    Смирнова Ольга Владимировна  0.580124  0.138380
4      Михайлов Андрей Сергеевич  0.369258  0.000000
6       Кузнецов Михаил Иванович  0.307209  0.000000
3    Попова Екатерина Николаевна  0.217559  0.014249
2  Соколова Анастасия Викторовна  0.208619  0.018714
0        Иванова Мария Сергеевна  0.140941  0.000000
1   Васильев Артем Александрович  0.012390  0.022310


# ============================================================================
# ШАГ 6: ФОРМИРОВАНИЕ EXCEL
# ==========================================

In [17]:
output_file = 'Prolongation_Report_Complete.xlsx'
wb = Workbook()

# Удаляем дефолтный лист
for sheet in wb.sheetnames:
    wb.remove(wb[sheet])

# Стили
beige_fill = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
orange_fill = PatternFill(start_color="FFD966", end_color="FFD966", fill_type="solid")
blue_fill = PatternFill(start_color="2E75B6", end_color="2E75B6", fill_type="solid")
green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
yellow_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
bold_font = Font(bold=True)
white_font = Font(bold=True, color="FFFFFF")
center_align = Alignment(horizontal='center', vertical='center')
left_align = Alignment(horizontal='left', vertical='center')
right_align = Alignment(horizontal='right', vertical='center')
money_format = '#,##0.00 ₽'
percent_format = '0.00%'
border = Border(left=Side(style='thin'), right=Side(style='thin'),
                top=Side(style='thin'), bottom=Side(style='thin'))

# ============= ЛИСТ 1: ВЕСЬ ОТДЕЛ =============
ws1 = wb.create_sheet("Весь отдел")

ws1.merge_cells('A1:A2')
ws1['A1'] = 'Месяц'
ws1['A1'].fill = beige_fill
ws1['A1'].font = bold_font
ws1['A1'].alignment = center_align
ws1['A1'].border = border

ws1.merge_cells('B1:D1')
ws1['B1'] = 'Пролонгации в первый месяц'
ws1['B1'].fill = beige_fill
ws1['B1'].font = bold_font
ws1['B1'].alignment = center_align
ws1['B1'].border = border

ws1.merge_cells('E1:G1')
ws1['E1'] = 'Пролонгации через месяц'
ws1['E1'].fill = beige_fill
ws1['E1'].font = bold_font
ws1['E1'].alignment = center_align
ws1['E1'].border = border

subs = ['', 'к пролонгации', 'пролонгировано', 'Коэффициент', 'к пролонгации', 'пролонгировано', 'Коэффициент']
for col, val in enumerate(subs, 1):
    if val:
        cell = ws1.cell(row=2, column=col, value=val)
        cell.fill = beige_fill
        cell.font = bold_font
        cell.alignment = center_align
        cell.border = border

for i, row in df_monthly.iterrows():
    r = 3 + i
    ws1.cell(row=r, column=1, value=row['Месяц'].replace(' 2023', ''))
    ws1.cell(row=r, column=2, value=row['coef1_den'])
    ws1.cell(row=r, column=3, value=row['coef1_num'])
    ws1.cell(row=r, column=4, value=row['coef1'])
    ws1.cell(row=r, column=5, value=row['coef2_den'])
    ws1.cell(row=r, column=6, value=row['coef2_num'])
    ws1.cell(row=r, column=7, value=row['coef2'])

    ws1.cell(row=r, column=1).alignment = center_align

    # КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: коэффициент должен быть в процентах!
    ws1.cell(row=r, column=4).number_format = percent_format  # ← ПРОЦЕНТЫ
    ws1.cell(row=r, column=7).number_format = percent_format  # ← ПРОЦЕНТЫ

    # Суммы - в денежном формате
    for col in [2, 3, 5, 6]:
        ws1.cell(row=r, column=col).number_format = money_format
        ws1.cell(row=r, column=col).alignment = right_align

    for col in range(1, 8):
        ws1.cell(row=r, column=col).border = border

for col in range(1, 8):
    ws1.column_dimensions[get_column_letter(col)].width = 18


# ============================================================================
# ЛИСТ 2: МЕНЕДЖЕРЫ ЗА ГОД (ИСПРАВЛЕННЫЙ)
# ============================================================================
ws2 = wb.create_sheet("Менеджеры за год")

ws2.merge_cells('A1:A2')
ws2['A1'] = 'Менеджер'
ws2['A1'].fill = beige_fill
ws2['A1'].font = bold_font
ws2['A1'].alignment = center_align
ws2['A1'].border = border

ws2.merge_cells('B1:D1')
ws2['B1'] = 'Пролонгации в первый месяц'
ws2['B1'].fill = beige_fill
ws2['B1'].font = bold_font
ws2['B1'].alignment = center_align
ws2['B1'].border = border

ws2.merge_cells('E1:G1')
ws2['E1'] = 'Пролонгации через месяц'
ws2['E1'].fill = beige_fill
ws2['E1'].font = bold_font
ws2['E1'].alignment = center_align
ws2['E1'].border = border

for col, val in enumerate(subs, 1):
    if val:
        cell = ws2.cell(row=2, column=col, value=val)
        cell.fill = beige_fill
        cell.font = bold_font
        cell.alignment = center_align
        cell.border = border

for i, row in df_mgr_yearly.iterrows():
    r = 3 + i
    ws2.cell(row=r, column=1, value=row['Менеджер'])
    ws2.cell(row=r, column=2, value=row['coef1_for'])
    ws2.cell(row=r, column=3, value=row['coef1_ren'])
    ws2.cell(row=r, column=4, value=row['coef1'])
    ws2.cell(row=r, column=5, value=row['coef2_for'])
    ws2.cell(row=r, column=6, value=row['coef2_ren'])
    ws2.cell(row=r, column=7, value=row['coef2'])

    ws2.cell(row=r, column=1).alignment = left_align

    # КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: коэффициент должен быть в процентах!
    ws2.cell(row=r, column=4).number_format = percent_format  # ← ПРОЦЕНТЫ
    ws2.cell(row=r, column=7).number_format = percent_format  # ← ПРОЦЕНТЫ

    # Суммы - в денежном формате
    for col in [2, 3, 5, 6]:
        ws2.cell(row=r, column=col).number_format = money_format
        ws2.cell(row=r, column=col).alignment = right_align

    for col in range(1, 8):
        ws2.cell(row=r, column=col).border = border

ws2.column_dimensions['A'].width = 35
for col in range(2, 8):
    ws2.column_dimensions[get_column_letter(col)].width = 18

# ============= ЛИСТ 3: МЕНЕДЖЕРЫ ПО МЕСЯЦАМ =============
ws3 = wb.create_sheet("Менеджеры по месяцам")

months_short = [m.replace(' 2023', '') for m in months_2023]
all_managers = sorted(df_mgr_yearly['Менеджер'].tolist())

# K1
ws3.merge_cells('A1:B1')
ws3['A1'] = 'Коэффициент 1'
ws3['A1'].fill = orange_fill
ws3['A1'].font = bold_font
ws3['A1'].alignment = center_align
ws3['A1'].border = border

ws3.merge_cells('A2:A3')
ws3['A2'] = 'Менеджер'
ws3['A2'].fill = beige_fill
ws3['A2'].font = bold_font
ws3['A2'].alignment = center_align
ws3['A2'].border = border

ws3.merge_cells('B2:M2')
ws3['B2'] = 'Месяц'
ws3['B2'].fill = beige_fill
ws3['B2'].font = bold_font
ws3['B2'].alignment = center_align
ws3['B2'].border = border

for col, month in enumerate(months_short, 2):
    cell = ws3.cell(row=3, column=col, value=month)
    cell.fill = beige_fill
    cell.font = bold_font
    cell.alignment = center_align
    cell.border = border

for i, mgr in enumerate(all_managers, 4):
    ws3.cell(row=i, column=1, value=mgr)
    ws3.cell(row=i, column=1).alignment = left_align
    ws3.cell(row=i, column=1).border = border
    for col, month in enumerate(months_2023, 2):
        val = manager_monthly_k1[month].get(mgr, 0)
        cell = ws3.cell(row=i, column=col, value=val)
        cell.number_format = percent_format
        cell.alignment = center_align
        cell.border = border
        if val >= 0.1:
            cell.fill = green_fill
        elif val >= 0.05:
            cell.fill = yellow_fill
        elif val > 0:
            cell.fill = red_fill

# K2
start_k2 = len(all_managers) + 6
ws3.merge_cells(f'A{start_k2}:B{start_k2}')
ws3[f'A{start_k2}'] = 'Коэффициент 2'
ws3[f'A{start_k2}'].fill = orange_fill
ws3[f'A{start_k2}'].font = bold_font
ws3[f'A{start_k2}'].alignment = center_align
ws3[f'A{start_k2}'].border = border

ws3.merge_cells(f'A{start_k2+1}:A{start_k2+2}')
ws3[f'A{start_k2+1}'] = 'Менеджер'
ws3[f'A{start_k2+1}'].fill = beige_fill
ws3[f'A{start_k2+1}'].font = bold_font
ws3[f'A{start_k2+1}'].alignment = center_align
ws3[f'A{start_k2+1}'].border = border

ws3.merge_cells(f'B{start_k2+1}:M{start_k2+1}')
ws3[f'B{start_k2+1}'] = 'Месяц'
ws3[f'B{start_k2+1}'].fill = beige_fill
ws3[f'B{start_k2+1}'].font = bold_font
ws3[f'B{start_k2+1}'].alignment = center_align
ws3[f'B{start_k2+1}'].border = border

for col, month in enumerate(months_short, 2):
    cell = ws3.cell(row=start_k2+2, column=col, value=month)
    cell.fill = beige_fill
    cell.font = bold_font
    cell.alignment = center_align
    cell.border = border

for i, mgr in enumerate(all_managers, start_k2 + 3):
    ws3.cell(row=i, column=1, value=mgr)
    ws3.cell(row=i, column=1).alignment = left_align
    ws3.cell(row=i, column=1).border = border
    for col, month in enumerate(months_2023, 2):
        val = manager_monthly_k2[month].get(mgr, 0)
        cell = ws3.cell(row=i, column=col, value=val)
        cell.number_format = percent_format
        cell.alignment = center_align
        cell.border = border
        if val > 0:
            cell.fill = green_fill

for col in range(1, 14):
    ws3.column_dimensions[get_column_letter(col)].width = 13 if col == 1 else 11

# ============= ЛИСТ 4: ГРАФИКИ И ВИЗУАЛИЗАЦИЯ =============
print("🎨 Создание графиков...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Аналитика пролонгации за 2023 год', fontsize=16, fontweight='bold')

# График 1: Динамика K1
ax1 = axes[0, 0]
months_names = [m.replace(' 2023', '') for m in df_monthly['Месяц']]
ax1.plot(months_names, df_monthly['coef1'] * 100, marker='o', linewidth=2, color='#2E75B6', markersize=8)
ax1.axhline(y=10, color='green', linestyle='--', alpha=0.7, label='Цель (10%)')
ax1.axhline(y=5, color='orange', linestyle='--', alpha=0.7, label='Минимум (5%)')
ax1.set_title('Динамика K1 по месяцам', fontsize=12, fontweight='bold')
ax1.set_ylabel('K1 (%)')
ax1.set_xlabel('Месяц')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, max(df_monthly['coef1'] * 100) + 5)

# График 2: Топ-10 менеджеров
ax2 = axes[0, 1]
top10 = df_mgr_yearly.head(10)
colors = ['#2E75B6' if v >= 0.1 else '#FFC000' if v >= 0.05 else '#ED7D31' for v in top10['coef1']]
bars = ax2.barh(top10['Менеджер'], top10['coef1'] * 100, color=colors)
ax2.set_title('Топ-10 менеджеров по годовому K1', fontsize=12, fontweight='bold')
ax2.set_xlabel('K1 (%)')
ax2.axvline(x=10, color='green', linestyle='--', alpha=0.7, label='Цель (10%)')
ax2.axvline(x=5, color='orange', linestyle='--', alpha=0.7, label='Минимум (5%)')
ax2.legend()
for bar, val in zip(bars, top10['coef1'] * 100):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)

# График 3: Сравнение сумм
ax3 = axes[1, 0]
x = range(len(months_names))
ax3.bar(x, df_monthly['coef1_den'] / 1000, width=0.4, label='База (тыс.₽)', color='#2E75B6', alpha=0.7)
ax3.bar(x, df_monthly['coef1_num'] / 1000, width=0.4, label='Пролонгации (тыс.₽)', color='#70AD47', alpha=0.7)
ax3.set_title('Суммы базы и пролонгаций по месяцам', fontsize=12, fontweight='bold')
ax3.set_ylabel('Тысячи рублей')
ax3.set_xlabel('Месяц')
ax3.set_xticks(x)
ax3.set_xticklabels(months_names, rotation=45)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# График 4: Круговая диаграмма
ax4 = axes[1, 1]
renewed = df_monthly['coef1_num'].sum()
not_renewed = df_monthly['coef1_den'].sum() - renewed
sizes = [renewed, not_renewed]
labels = [f'Пролонгировано\n{renewed/1000:.0f}K ₽', f'Не пролонгировано\n{not_renewed/1000:.0f}K ₽']
colors_pie = ['#70AD47', '#ED7D31']
ax4.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%', startangle=90)
ax4.set_title('Доля пролонгированных сумм за год', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('dashboard_charts.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()

ws4 = wb.create_sheet("Графики и дашборд")
img = XLImage('dashboard_charts.png')
img.width = 800
img.height = 600
ws4.add_image(img, 'A1')
ws4.column_dimensions['A'].width = 100
ws4.row_dimensions[1].height = 450

# ============= ЛИСТ 5: АНАЛИТИЧЕСКИЙ ОТЧЕТ =============
ws5 = wb.create_sheet("Аналитический отчет")

ws5.merge_cells('A1:G1')
ws5['A1'] = 'АНАЛИТИЧЕСКИЙ ОТЧЕТ ПО ПРОЛОНГАЦИИ ДОГОВОРОВ ЗА 2023 ГОД'
ws5['A1'].font = Font(bold=True, size=14, color="FFFFFF")
ws5['A1'].fill = blue_fill
ws5['A1'].alignment = center_align
ws5['A1'].border = border

ws5['A2'] = f'Дата формирования: {datetime.now().strftime("%d.%m.%Y")}'
ws5['A2'].font = Font(italic=True)

# Ключевые выводы
ws5.merge_cells('A4:G4')
ws5['A4'] = '📌 1. КЛЮЧЕВЫЕ ВЫВОДЫ'
ws5['A4'].font = bold_font
ws5['A4'].fill = orange_fill
ws5['A4'].border = border

total_base = df_monthly['coef1_den'].sum()
total_renewed = df_monthly['coef1_num'].sum()
yearly_k1 = total_renewed / total_base if total_base > 0 else 0
yearly_k2 = df_monthly['coef2_num'].sum() / df_monthly['coef2_den'].sum() if df_monthly['coef2_den'].sum() > 0 else 0

conclusions = [
    f'• Годовой коэффициент пролонгации (K1) по отделу: {yearly_k1:.2%}',
    f'• Годовой коэффициент пролонгации (K2) по отделу: {yearly_k2:.2%}',
    f'• Всего завершено проектов: {len(project_last_month):,}',
    f'• Общая сумма базы: {total_base:,.0f} ₽',
    f'• Общая сумма пролонгаций: {total_renewed:,.0f} ₽',
    f'• Лучший месяц: {df_monthly.loc[df_monthly["coef1"].idxmax(), "Месяц"].replace(" 2023", "")} ({df_monthly["coef1"].max():.2%})',
    f'• Лучший менеджер: {df_mgr_yearly.iloc[0]["Менеджер"]} ({df_mgr_yearly.iloc[0]["coef1"]:.2%})',
]

for i, concl in enumerate(conclusions, 5):
    ws5[f'A{i}'] = concl
    ws5.merge_cells(f'A{i}:G{i}')
    ws5[f'A{i}'].border = border

# Месячная динамика
row = 14
ws5.merge_cells(f'A{row}:G{row}')
ws5[f'A{row}'] = '📈 2. МЕСЯЧНАЯ ДИНАМИКА K1'
ws5[f'A{row}'].font = bold_font
ws5[f'A{row}'].fill = orange_fill
ws5[f'A{row}'].border = border
row += 1

headers = ['Месяц', 'K1', 'Оценка', 'Сумма базы', 'Сумма пролонгаций']
for col, h in enumerate(headers, 1):
    cell = ws5.cell(row=row, column=col, value=h)
    cell.fill = beige_fill
    cell.font = bold_font
    cell.alignment = center_align
    cell.border = border
row += 1

for i, r in df_monthly.iterrows():
    ws5.cell(row=row, column=1, value=r['Месяц'].replace(' 2023', ''))
    ws5.cell(row=row, column=2, value=r['coef1'])
    ws5.cell(row=row, column=4, value=r['coef1_den'])
    ws5.cell(row=row, column=5, value=r['coef1_num'])

    ws5.cell(row=row, column=2).number_format = percent_format
    ws5.cell(row=row, column=4).number_format = money_format
    ws5.cell(row=row, column=5).number_format = money_format

    if r['coef1'] >= 0.1:
        ws5.cell(row=row, column=3, value="🟢 Отлично")
        ws5.cell(row=row, column=3).fill = green_fill
    elif r['coef1'] >= 0.05:
        ws5.cell(row=row, column=3, value="🟡 Хорошо")
        ws5.cell(row=row, column=3).fill = yellow_fill
    else:
        ws5.cell(row=row, column=3, value="🔴 Требует внимания")
        ws5.cell(row=row, column=3).fill = red_fill

    for col in range(1, 6):
        ws5.cell(row=row, column=col).border = border
        if col != 1:
            ws5.cell(row=row, column=col).alignment = center_align
    row += 1

# Рекомендации
row += 2
ws5.merge_cells(f'A{row}:G{row}')
ws5[f'A{row}'] = '💡 3. РЕКОМЕНДАЦИИ ДЛЯ РУКОВОДИТЕЛЯ'
ws5[f'A{row}'].font = bold_font
ws5[f'A{row}'].fill = orange_fill
ws5[f'A{row}'].border = border
row += 1

recommendations = [
    f'1. Внедрить систему напоминаний о пролонгации за 60 и 30 дней до завершения договора',
    f'2. Провести обучение для менеджеров с K1 ниже среднего (цель: достичь 10%)',
    f'3. Разработать скрипты работы с клиентами для повышения K2',
    f'4. Наградить лучшего менеджера: {df_mgr_yearly.iloc[0]["Менеджер"]}',
    f'5. Ежеквартально проводить анализ динамики коэффициентов',
]

for i, rec in enumerate(recommendations, row):
    ws5[f'A{i}'] = rec
    ws5.merge_cells(f'A{i}:G{i}')
    ws5[f'A{i}'].border = border

for col in range(1, 8):
    ws5.column_dimensions[get_column_letter(col)].width = 22

# ============= ЛИСТ 6: КЛЮЧЕВЫЕ МЕТРИКИ =============
ws6 = wb.create_sheet("Ключевые метрики")

ws6.merge_cells('A1:F1')
ws6['A1'] = '📊 КЛЮЧЕВЫЕ МЕТРИКИ ПРОЛОНГАЦИИ ЗА 2023 ГОД'
ws6['A1'].font = Font(bold=True, size=14, color="FFFFFF")
ws6['A1'].fill = blue_fill
ws6['A1'].alignment = center_align
ws6['A1'].border = border

metrics = [
    ('Всего завершённых проектов', f"{len(project_last_month):,}", 'шт.'),
    ('Активных менеджеров', f"{len(df_mgr_yearly):,}", 'чел.'),
    ('Общая сумма базы (K1)', f"{total_base:,.0f}", '₽'),
    ('Общая сумма пролонгаций (K1)', f"{total_renewed:,.0f}", '₽'),
    ('Годовой K1', f"{yearly_k1:.2%}", ''),
    ('Годовой K2', f"{yearly_k2:.2%}", ''),
    ('Лучший месяц по K1', df_monthly.loc[df_monthly['coef1'].idxmax(), 'Месяц'].replace(' 2023', ''),
     f"({df_monthly['coef1'].max():.2%})"),
    ('Лучший менеджер по K1', df_mgr_yearly.iloc[0]['Менеджер'],
     f"({df_mgr_yearly.iloc[0]['coef1']:.2%})"),
]

row = 3
for metric in metrics:
    ws6.cell(row=row, column=1, value=metric[0]).font = bold_font
    ws6.cell(row=row, column=2, value=metric[1]).alignment = right_align
    ws6.cell(row=row, column=3, value=metric[2])
    for col in range(1, 4):
        ws6.cell(row=row, column=col).border = border
    row += 1

ws6.column_dimensions['A'].width = 30
ws6.column_dimensions['B'].width = 25
ws6.column_dimensions['C'].width = 10

# ============= СОХРАНЕНИЕ =============
wb.save(output_file)
print(f"\n✅ Отчёт сохранён: {output_file}")

from google.colab import files
files.download(output_file)

print("\n" + "=" * 70)
print("📊 ИТОГОВЫЙ ОТЧЕТ ГОТОВ")
print("=" * 70)
print("""
Структура отчета:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 📋 Весь отдел - помесячные данные (K1 и K2 в %)
2. 👥 Менеджеры за год - все менеджеры с годовыми KPI
3. 📊 Менеджеры по месяцам - матрица K1 и K2 с цветовой индикацией
4. 📈 Графики и дашборд - визуализация основных метрик
5. 📄 Аналитический отчет - выводы и рекомендации
6. 🎯 Ключевые метрики - сводка KPI""")

🎨 Создание графиков...

✅ Отчёт сохранён: Prolongation_Report_Complete.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📊 ИТОГОВЫЙ ОТЧЕТ ГОТОВ

Структура отчета:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 📋 Весь отдел - помесячные данные (K1 и K2 в %)
2. 👥 Менеджеры за год - все менеджеры с годовыми KPI
3. 📊 Менеджеры по месяцам - матрица K1 и K2 с цветовой индикацией
4. 📈 Графики и дашборд - визуализация основных метрик
5. 📄 Аналитический отчет - выводы и рекомендации
6. 🎯 Ключевые метрики - сводка KPI
